# 01 — Data Preparation (Run this notebook first)

This notebook prepares the Roboflow COCO export for the **two-stage thesis pipeline**:
1) stage-1 DeiT classifier inputs, and 2) stage-2 Faster R-CNN detection inputs.

**Expected output:** `config.json`, `dataset/train|valid|test`, and `results/dataset_qa_summary.csv`.


In [1]:
# Run this cell first: environment setup + safety checks (Kaggle primary, Colab secondary)
import os, sys, json, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ
ENV_NAME = "kaggle" if IS_KAGGLE else ("colab" if IS_COLAB else "local")

if IS_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
elif IS_COLAB:
    WORK_DIR = Path("/content")
else:
    WORK_DIR = Path.cwd()
WORK_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = WORK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV_NAME}")
print(f"WORK_DIR: {WORK_DIR}")


GPU check
GPU: Tesla T4
PyTorch: 2.10.0+cu128
CUDA: OK
GPU: Tesla T4
PyTorch: 2.10.0+cu128
CUDA: OK
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 121.9 MB/s eta 0:00:00
Done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
# Shared config for downstream notebooks (02, 03)
from pathlib import Path
import json

config = {
    "experiment_name": "Exp1_deitSmallDistilled_t05",
    "seed": SEED,
    "deit_variant": "facebook/deit-small-distilled-patch16-224",
    "confidence_threshold": 0.5,
    "environment": ENV_NAME,
    "work_dir": str(WORK_DIR),
    "dataset_root": str(WORK_DIR / "dataset"),
    "results_dir": str(RESULTS_DIR),
    "splits": {"train": 0.70, "valid": 0.15, "test": 0.15},
    "paths": {
        "train_images": str(WORK_DIR / "dataset/train"),
        "valid_images": str(WORK_DIR / "dataset/valid"),
        "test_images": str(WORK_DIR / "dataset/test"),
    },
    "deit": {"epochs": 10, "batch_size": 16, "lr": 3e-5, "enable_5fold_cv": False},
    "frcnn": {"epochs": 20, "batch_size": 4, "lr": 1e-4}
}

cfg_path = WORK_DIR / "config.json"
with open(cfg_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved config: {cfg_path}")


Config saved: /kaggle/working/config.json


In [3]:
# Roboflow download (secure key handling)
import os
from pathlib import Path

rf_api_key = None
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        rf_api_key = UserSecretsClient().get_secret("rf_api_key")
    except Exception:
        rf_api_key = None
if not rf_api_key:
    rf_api_key = os.getenv("ROBOFLOW_API_KEY", "")

if not rf_api_key:
    raise RuntimeError("Missing Roboflow API key. Add Kaggle secret 'rf_api_key' or set ROBOFLOW_API_KEY.")

from roboflow import Roboflow
rf = Roboflow(api_key=rf_api_key)
# Update these two values for your workspace/project:
workspace_slug = os.getenv("ROBOFLOW_WORKSPACE", "")
project_slug = os.getenv("ROBOFLOW_PROJECT", "")
version = int(os.getenv("ROBOFLOW_VERSION", "1"))
if not workspace_slug or not project_slug:
    raise RuntimeError("Set ROBOFLOW_WORKSPACE and ROBOFLOW_PROJECT env vars before running download cell.")

raw_dir = WORK_DIR / "raw_coco"
raw_dir.mkdir(parents=True, exist_ok=True)
project = rf.workspace(workspace_slug).project(project_slug)
dataset = project.version(version).download("coco", location=str(raw_dir))
RAW_DIR = Path(dataset.location)
print(f"Downloaded Roboflow COCO to: {RAW_DIR}")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Find-infected-and-healthy-2 in coco:: 100%|██████████| 468/468 [00:00<00:00, 7046.29it/s]

Downloaded version 2
Raw dir: /kaggle/working/Find-infected-and-healthy-2


In [4]:
# Build 70/15/15 split with annotation preservation + QA summary
import json, shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd

with open(WORK_DIR / "config.json") as f:
    config = json.load(f)

ann_file = next(iter(list(Path(RAW_DIR).glob("**/_annotations.coco.json"))), None)
if ann_file is None:
    raise RuntimeError("Could not find _annotations.coco.json in downloaded Roboflow folder.")

with open(ann_file) as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]
categories = coco["categories"]
ann_by_img = {}
for a in annotations:
    ann_by_img.setdefault(a["image_id"], []).append(a)

def infer_label(img):
    anns = ann_by_img.get(img["id"], [])
    return "infected" if len(anns) > 0 else "healthy"

ids = [img["id"] for img in images]
labels = [infer_label(img) for img in images]
train_ids, tmp_ids, y_train, y_tmp = train_test_split(ids, labels, test_size=0.30, random_state=SEED, stratify=labels)
valid_ids, test_ids = train_test_split(tmp_ids, test_size=0.50, random_state=SEED, stratify=y_tmp)

dataset_root = Path(config["dataset_root"])
for split, split_ids in {"train": train_ids, "valid": valid_ids, "test": test_ids}.items():
    out_dir = dataset_root / split
    out_dir.mkdir(parents=True, exist_ok=True)
    split_images = [img for img in images if img["id"] in set(split_ids)]
    split_anns = [a for a in annotations if a["image_id"] in set(split_ids)]
    coco_out = {"images": split_images, "annotations": split_anns, "categories": categories}
    with open(out_dir / "_annotations.coco.json", "w") as f:
        json.dump(coco_out, f)
    for img in split_images:
        src = Path(RAW_DIR) / img["file_name"]
        if src.exists():
            (out_dir / src.name).write_bytes(src.read_bytes())

qa = []
for split in ["train", "valid", "test"]:
    with open(dataset_root / split / "_annotations.coco.json") as f:
        c = json.load(f)
    ids_split = {i['id'] for i in c['images']}
    anns = c['annotations']
    infected_ids = {a['image_id'] for a in anns}
    qa.append({
        "split": split,
        "num_images": len(c["images"]),
        "num_healthy": len(ids_split - infected_ids),
        "num_infected": len(infected_ids),
        "num_bboxes": len(anns)
    })

df = pd.DataFrame(qa)
df.to_csv(RESULTS_DIR / "dataset_qa_summary.csv", index=False)
print(df)
print("Saved:", RESULTS_DIR / "dataset_qa_summary.csv")
print("Expected output ready:", dataset_root)


Split       Images  Healthy  Infected
----------------------------------------
train            324      189      135
valid             69       40       29
test              70       42       28

Data QA
Split       Images  Healthy  Infected
----------------------------------------
train            324      196      127
valid             69       42       27
test              70       43       27

Done. Publish Kaggle dataset: config.json + dataset folder
Input path: /kaggle/input/datasets/nikachuu/data-prep
